In [1]:
tabla='qbefi10'
dim='dim_cqxestfis'

In [2]:
import re

cadena = """
    estfiscod character(1) COLLATE pg_catalog."default",
    estfisdes character(70) COLLATE pg_catalog."default",
    estfisdescor character(5) COLLATE pg_catalog."default"
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ','.join(nombrecitos)

print(insertado_col)

ALTER COLUMN estfiscod TYPE character(1),
ALTER COLUMN estfisdes TYPE character(70),
ALTER COLUMN estfisdescor TYPE character(5);

estfiscod,estfisdes,estfisdescor


In [3]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
import decouple

config = decouple.AutoConfig(' ')

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [4]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527
engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)
connection2.close()

In [5]:
base2.head()

,estfiscod,estfisdes,estfisdescor
0,1,(I) PACIENTE SANO,I
1,7,** NO UTILIZAR **,E
2,2,(II) PACIENTE CON ENFERMEDAD SISTEMICA LEVE,II
3,3,(III) PACIENTE CON ENFERMEDAD SISTEMICA SEVERA,III
4,4,(IV) PACIENTE CON ENFERMEDAD SISTEMICA SEVERA ...,IV


In [6]:
base2.columns

Index(['estfiscod', 'estfisdes', 'estfisdescor'], dtype='object')

In [7]:
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla qbefi10 antes de la inserción: 7


/tmp/ipykernel_1168123/3136215966.py:4: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  cant_antes = connection3.execute(query_count_before).fetchone()[0]


In [8]:

base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

7

In [9]:
query=f"""
ALTER TABLE tmp_{tabla}

{cadena_alter}
UPDATE {tabla} b

SET

estfisdes = t.estfisdes,
estfisdescor = t.estfisdescor

FROM tmp_{tabla} t 
WHERE 

b.estfiscod = t.estfiscod and b.estfiscod IS NOT NULL;

INSERT INTO {tabla}

({insertado_col})
SELECT 
{insertado_col}

FROM tmp_{tabla}  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {tabla} b 
    WHERE 
    
    b.estfiscod = t.estfiscod and b.estfiscod IS NOT NULL
) ;
"""

c1= text(query)
connection3.execute(c1)

In [10]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""


c2= text(query2)
cursor=connection3.execute(c2)


Cantidad de filas en la tabla qbefi10 después de la inserción: 7
La cantidad de filas insertadas fue de: 0


In [ ]:
query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

In [ ]:
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection3)

In [ ]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

In [ ]:
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

In [ ]:
query=f"""

ALTER TABLE tmp_{tabla} 
{cadena_alter}
INSERT INTO {dim} 

(cod_efi,des_efi,dco_efi) 

SELECT 

estfiscod,estfisdes,estfisdescor

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d
    WHERE 
    
    d.cod_efi = t.estfiscod

);
"""

c1= text(query)
cursor=connection4.execute(c1)


In [ ]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

In [ ]:
c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

In [11]:
connection3.close()
connection4.close()